# Clase 12 · Notebook 02
# Regularización: dropout, batch norm y early stopping

**Introducción al Deep Learning — Módulo 2, Unidad 1: Redes de neuronas (Parte II)**

Vamos a **provocar sobreajuste** a propósito (una red grande con pocos datos) y luego a combatirlo con
técnicas de regularización, comparando las **curvas de aprendizaje** (entrenamiento vs validación).

1. Provocar sobreajuste y verlo en las curvas.
2. Añadir **dropout** + **batch normalization**.
3. Añadir **early stopping**.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos (pocos ejemplos para favorecer el sobreajuste)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data); y = data.target
# Usamos POCOS datos de entrenamiento para que sea fácil sobreajustar
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.85, random_state=42, stratify=y)
print("Entrenamiento (pequeño):", Xtr.shape[0], "| Test:", Xte.shape[0])

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

def curvas(h, titulo):
    plt.figure(figsize=(7, 4))
    plt.plot(h.history["loss"], label="entrenamiento", color="#000F9F")
    plt.plot(h.history["val_loss"], label="validación", color="#FF647E")
    plt.title(titulo); plt.xlabel("Época"); plt.ylabel("loss")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. Red grande SIN regularización: sobreajuste

Veremos que la pérdida de **entrenamiento** baja mucho, pero la de **validación** se queda atrás o sube:
señal de sobreajuste.

In [ ]:
def red_basica():
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(X.shape[1],)),
                    Dense(128, activation="relu"),
                    Dense(128, activation="relu"),
                    Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

h_over = red_basica().fit(Xtr, ytr, epochs=120, batch_size=8, validation_split=0.3, verbose=0)
curvas(h_over, "Sin regularización (sobreajuste)")
print("Brecha train/val (val_loss - loss) al final: %.3f" %
      (h_over.history["val_loss"][-1] - h_over.history["loss"][-1]))

## 3. CON dropout y batch normalization

El **dropout** apaga neuronas al azar y la **batch normalization** estabiliza las activaciones. La brecha
entre entrenamiento y validación debería reducirse.

In [ ]:
def red_regularizada():
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(X.shape[1],)),
                    Dense(128, activation="relu"), BatchNormalization(), Dropout(0.5),
                    Dense(128, activation="relu"), BatchNormalization(), Dropout(0.5),
                    Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

h_reg = red_regularizada().fit(Xtr, ytr, epochs=120, batch_size=8, validation_split=0.3, verbose=0)
curvas(h_reg, "Con dropout + batch normalization")
print("Brecha train/val al final: %.3f" %
      (h_reg.history["val_loss"][-1] - h_reg.history["loss"][-1]))

## 4. Early stopping: parar a tiempo

Detenemos el entrenamiento cuando la pérdida de validación deja de mejorar, y recuperamos los mejores pesos.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
modelo = red_regularizada()
h_es = modelo.fit(Xtr, ytr, epochs=200, batch_size=8, validation_split=0.3, verbose=0, callbacks=[es])

print("Épocas ejecutadas:", len(h_es.history["loss"]), "(se detuvo solo)")
print("Accuracy en test: %.1f %%" % (modelo.evaluate(Xte, yte, verbose=0)[1] * 100))
curvas(h_es, "Con early stopping")

El early stopping evita seguir entrenando cuando ya no mejora, ahorrando tiempo y reduciendo el sobreajuste.

## Experimenta tú

1. Cambia el `Dropout(0.5)` a 0.2 o 0.8 y observa el efecto.
2. Quita la `BatchNormalization` y compara.
3. Cambia `patience` del early stopping.

## Resumen

- El **sobreajuste** se ve cuando la validación empeora mientras el entrenamiento mejora.
- **Dropout** y **batch normalization** reducen la brecha y estabilizan el entrenamiento.
- **Early stopping** detiene el entrenamiento en el mejor momento y restaura los mejores pesos.
- Diagnostica siempre con las **curvas de aprendizaje**.